In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch

def run_inference_on_sample(npz_file_path, model, sequence_length=450, fs=30.0, pad_margin=30):
    """
    Execute inference on an NPZ sample, applying reflection padding prior to 
    bandpass filtering to completely suppress convolution boundary ringing at frame 0.
    """
    print(f"--- INFERENCE RUN ON: {os.path.basename(npz_file_path)} ---")
    data = load_npz_file(npz_file_path)
    if data is None:
        return None, None
    
    ppg_key = 'ppg_values' if 'ppg_values' in data else 'ppg'
    if len(data[ppg_key]) < sequence_length:
        print("Error: Input file length is shorter than required sequence length.")
        return None, None
        
    # 1. Ground Truth Pipeline (Padded filtering to prevent edge artifacts)
    raw_target_ppg = data[ppg_key][:sequence_length].copy()
    gt_padded = np.pad(raw_target_ppg, pad_margin, mode='reflect')
    gt_filtered = butter_bandpass_filter(gt_padded - np.mean(gt_padded), fs=fs)
    gt_cropped = gt_filtered[pad_margin:-pad_margin]
    gt_norm = (gt_cropped - np.mean(gt_cropped)) / (np.std(gt_cropped) + 1e-8)
    
    # 2. Multi-ROI Input Tensor Preparation
    roi_inputs = []
    for roi_name in ROI_ORDER:
        actual_key = next((k for k in data.files if roi_name == k), None)
        if actual_key is not None:
            roi_data = data[actual_key][:sequence_length]
        else:
            roi_data = np.zeros((sequence_length, 24, 24, 3), dtype=np.uint8)
        roi_inputs.append(roi_data.astype(np.float32) / 255.0)
        
    stacked_rois = np.stack(roi_inputs, axis=0)
    stacked_rois = np.transpose(stacked_rois, (0, 4, 1, 2, 3))
    stacked_rois = np.reshape(stacked_rois, (30, sequence_length, 24, 24))
    
    input_tensor = torch.tensor(stacked_rois, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    
    # 3. Network Forward Pass
    model.eval()
    with torch.no_grad():
        output_signal = model(input_tensor).cpu().numpy()[0]
        
    # 4. Prediction Pipeline: Pad -> Filter -> Crop -> Z-Score Normalize
    pred_padded = np.pad(output_signal, pad_margin, mode='reflect')
    pred_filtered = butter_bandpass_filter(pred_padded - np.mean(pred_padded), fs=fs)
    pred_cropped = pred_filtered[pad_margin:-pad_margin]
    pred_norm = (pred_cropped - np.mean(pred_cropped)) / (np.std(pred_cropped) + 1e-8)
    
    # 5. Heart Rate Metrics Calculation
    hr_pred = calculate_bpm_from_fft(pred_norm, fs=fs)
    hr_true = calculate_bpm_from_fft(gt_norm, fs=fs)
    
    print(f"  Predicted HR  : {hr_pred:.2f} BPM")
    print(f"  Ground-Truth  : {hr_true:.2f} BPM")
    print(f"  Absolute Error: {abs(hr_pred - hr_true):.2f} BPM")
    print("----------------------------------------------------\n")
    return pred_norm, gt_norm

In [ ]:
# ============================================================================
# EXECUTION & DASHBOARD PLOTTING CELL
# ============================================================================

# Select sample file from validation split
sample_val_file = val_split[0] if len(val_split) > 0 else npz_files[0]
inference_pred, inference_gt = run_inference_on_sample(sample_val_file, model)

# Create 2x2 Visualization Dashboard
fig, axs = plt.subplots(2, 2, figsize=(15, 11))

# --- Subplot 1: Convergence Curves (DUAL AXIS) ---
epochs_range = range(1, len(train_losses) + 1)
ax1 = axs[0, 0]
line1 = ax1.plot(epochs_range, train_losses, label='Train Loss', color='#1f77b4', linewidth=2.5, marker='o', markersize=3)
line2 = ax1.plot(epochs_range, val_losses, label='Val Loss', color='#ff7f0e', linewidth=2.5, marker='s', markersize=3)
ax1.set_title('Model Convergence (Loss vs. Validation PCC)')
ax1.set_xlabel('Training Epochs')
ax1.set_ylabel('Tri-Objective Loss (Dynamic Weights)', color='#333333')
ax1.tick_params(axis='y', labelcolor='#333333')

# Add PCC on secondary Y axis
ax2 = ax1.twinx()
line3 = ax2.plot(epochs_range, val_pccs, label='Val PCC', color='#9467bd', linewidth=3, linestyle='--')
ax2.set_ylabel('Pearson Correlation Coefficient (Higher is Better)', color='#9467bd', fontweight='bold')
ax2.tick_params(axis='y', labelcolor='#9467bd')

# Combine legends from both axes
lines = line1 + line2 + line3
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left')

# --- Subplot 2: Standardized Waveform Alignment ---
if inference_pred is not None and inference_gt is not None:
    axs[0, 1].plot(inference_gt, label='GT PPG (Normalized μ=0, σ=1)', color='#7f7f7f', alpha=0.85, linewidth=2)
    axs[0, 1].plot(inference_pred, label='10-ROI Predicted rPPG', color='#2ca02c', linewidth=2.2)
    axs[0, 1].set_title('Signal Waveform Comparison (Inference Showcase)')
    axs[0, 1].set_xlabel('Frames Timeline')
    axs[0, 1].set_ylabel('Normalized Amplitude (Z-Score)')
    axs[0, 1].set_ylim(-3.5, 3.5)  # Constrain y-axis to valid standardized bounds
    axs[0, 1].legend(loc='upper right')

# --- Subplot 3: Spectral Density Alignment ---
if inference_pred is not None and inference_gt is not None:
    freqs_p, psd_p = welch(inference_pred, fs=30.0, nperseg=len(inference_pred))
    freqs_t, psd_t = welch(inference_gt, fs=30.0, nperseg=len(inference_gt))
    
    axs[1, 0].plot(freqs_t * 60.0, psd_t, label='GT Spectrum', color='#7f7f7f', alpha=0.85, linewidth=2)
    axs[1, 0].plot(freqs_p * 60.0, psd_p, label='Pred Spectrum', color='#d62728', linewidth=2.2)
    axs[1, 0].set_xlim(40, 180)
    axs[1, 0].set_title('Power Spectral Density (PSD) Comparison')
    axs[1, 0].set_xlabel('Heart Rate (BPM)')
    axs[1, 0].set_ylabel('Spectral Magnitude')
    axs[1, 0].legend(loc='upper right')

# --- Subplot 4: Summary Card ---
axs[1, 1].axis('off')
metric_summary_card = (
    "10-ROI Spatio-Temporal Evaluation Summary\n\n"
    f"\u2022 Pearson Correlation Coefficient (PCC): {metrics['PCC']:.4f}\n"
    f"\u2022 Mean Absolute Error (MAE): {metrics['MAE']:.2f} BPM\n"
    f"\u2022 Root Mean Squared Error (RMSE): {metrics['RMSE']:.2f} BPM\n"
    f"\u2022 Signal-to-Noise Ratio (SNR): {metrics['SNR']:.2f} dB\n\n"
    f"Architecture & Configuration:\n"
    f"\u2022 Spatio-Temporal Conv3D + Deep 1D Temporal Encoder\n"
    f"\u2022 Input Channels: 30 (10 Face Regions \u00d7 3 RGB)\n"
    f"\u2022 Forward Normalization: AC/DC [ (x - \u03bc) / (\u03bc + \u03b5) ]\n"
    f"\u2022 Sequence segment window: 450 frames (15.0s @ 30 FPS)\n"
    f"\u2022 Loss Strategy: Dynamic Weighting (Phase + Freq + Amplitude)"
)
axs[1, 1].text(0.05, 0.5, metric_summary_card, fontsize=11, va='center',
              bbox=dict(boxstyle='round,pad=1.5', facecolor='#f8f9fa', edgecolor='#dee2e6'))

plt.tight_layout()
plt.show()